# vLLM Colab Server

Google Colab GPU'sunda vLLM ile model serve et, Cloudflare Tunnel ile `api.ersamely.com` üzerinden eriş.

## Colab Secrets (gerekli)
- `HF_TOKEN` — HuggingFace erişimi
- `CF_TUNNEL_TOKEN` — Cloudflare named tunnel
- `VLLM_API_KEY` — API erişim anahtarı

## Kullanım
1. **A**: Kurulum (bir kere)
2. **B**: Model seç + başlat
3. **C**: Tunnel bağla
4. **Model değiştir**: B'ye dön, ACTIVE_MODEL'i değiştir, tekrar çalıştır

---
# A) İlk Kurulum (bir kere)

In [ ]:
import os
import subprocess

os.environ['VLLM_USE_FLASHINFER'] = '0'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    subprocess.run(
        ['huggingface-cli', 'login', '--token', hf_token, '--add-to-git-credential'],
        capture_output=True, text=True, check=True
    )
    print('\u2705 HuggingFace login')
except KeyError:
    print('\u26a0\ufe0f HF_TOKEN tanımlı değil')
except Exception as e:
    print(f'\u26a0\ufe0f HF login atlandı: {e}')

!pip -q install -U vllm httpx huggingface_hub 2>&1 | tail -5
!pip uninstall flashinfer flashinfer-python -y 2>/dev/null

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

!nvidia-smi | head -12

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

LOG_DIR = '/content/logs'
os.makedirs(LOG_DIR, exist_ok=True)

---
# B) Model Seç ve Başlat

`ACTIVE_MODEL` değiştirip bu bölümü tekrar çalıştırman yeterli.
Tunnel (C bölümü) ayakta kalır, kesinti olmaz.

In [ ]:
from google.colab import userdata

# ╔══════════════════════════════════════════════════════════════════╗
# ║  MODEL KATALOĞu — Buraya modelleri ekle/çıkar                 ║
# ╚══════════════════════════════════════════════════════════════════╝
MODELS = {
    'qwen3.6-35b': {
        'repo': 'Qwen/Qwen3.6-35B-A3B',
        'max_model_len': 65536,
        'extra_args': ['--reasoning-parser', 'qwen3'],
    },
    'qwen3.6-35b-fp8': {
        'repo': 'Qwen/Qwen3.6-35B-A3B-FP8',
        'max_model_len': 65536,
        'extra_args': ['--reasoning-parser', 'qwen3'],
    },
    'qwen3-32b': {
        'repo': 'Qwen/Qwen3-32B',
        'max_model_len': 32768,
        'extra_args': ['--reasoning-parser', 'qwen3'],
    },
    'qwen3-32b-awq': {
        'repo': 'Qwen/Qwen3-32B-AWQ',
        'max_model_len': 32768,
        'extra_args': ['--quantization', 'awq', '--reasoning-parser', 'qwen3'],
    },
    'qwen3-8b': {
        'repo': 'Qwen/Qwen3-8B',
        'max_model_len': 65536,
        'extra_args': ['--reasoning-parser', 'qwen3'],
    },
    'deepseek-r1-qwen-32b': {
        'repo': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B',
        'max_model_len': 32768,
        'extra_args': [],
    },
}

# ╔══════════════════════════════════════════════════════════════════╗
# ║  AKTİF MODEL — Bunu değiştir, B bölümünü tekrar çalıştır      ║
# ╚══════════════════════════════════════════════════════════════════╝
ACTIVE_MODEL = 'qwen3.6-35b'

# ─── Resolve ───
if ACTIVE_MODEL not in MODELS:
    print('Mevcut modeller:')
    for name in MODELS:
        print(f'  - {name}')
    raise ValueError(f'Model bulunamadı: {ACTIVE_MODEL}')

_m = MODELS[ACTIVE_MODEL]
MODEL_REPO    = _m['repo']
MAX_MODEL_LEN = _m.get('max_model_len', 32768)
GPU_MEM_UTIL  = _m.get('gpu_mem_util', 0.98)
EXTRA_ARGS    = _m.get('extra_args', [])
VLLM_PORT     = 8090

API_KEY       = userdata.get('VLLM_API_KEY')
VLLM_BASE_URL = f'http://localhost:{VLLM_PORT}'
TUNNEL_URL    = 'https://api.ersamely.com'

print(f'\u250c{"\u2500"*50}\u2510')
print(f'\u2502 {ACTIVE_MODEL:48s} \u2502')
print(f'\u251c{"\u2500"*50}\u2524')
print(f'\u2502  repo:      {MODEL_REPO[:36]:36s} \u2502')
print(f'\u2502  max_len:   {MAX_MODEL_LEN:<36} \u2502')
print(f'\u2502  gpu_util:  {GPU_MEM_UTIL:<36} \u2502')
print(f'\u2502  extra:     {str(EXTRA_ARGS)[:36]:36s} \u2502')
print(f'\u2514{"\u2500"*50}\u2518')

In [ ]:
import subprocess
import time

import requests
import torch

LOG_PATH = f'{LOG_DIR}/vllm.log'

# Önceki vLLM'i kapat, GPU temizle
print(f'\U0001f6d1 Önceki vLLM durduruluyor...')
subprocess.run(['pkill', '-f', 'vllm serve'], capture_output=True)
time.sleep(3)
torch.cuda.empty_cache()

# Yeni model başlat
env = {**os.environ, 'VLLM_USE_FLASHINFER': '0'}
cmd = [
    'vllm', 'serve', MODEL_REPO,
    '--host', '0.0.0.0',
    '--port', str(VLLM_PORT),
    '--max-model-len', str(MAX_MODEL_LEN),
    '--gpu-memory-utilization', str(GPU_MEM_UTIL),
    '--trust-remote-code',
    '--api-key', API_KEY,
    *EXTRA_ARGS,
]

with open(LOG_PATH, 'w') as f:
    proc = subprocess.Popen(
        cmd, stdout=f, stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, env=env
    )

print(f'\U0001f680 {ACTIVE_MODEL} başlatıldı (PID: {proc.pid})')


def get_log_tail(n=5):
    try:
        with open(LOG_PATH) as f:
            return ''.join(f.readlines()[-n:])
    except OSError:
        return ''


def detect_phase(log):
    lower = log.lower()
    if 'error' in lower or 'traceback' in lower:
        return '\u274c HATA'
    if 'downloading' in lower or 'fetching' in lower:
        return '\U0001f4e5 İndiriliyor'
    if 'loading model' in lower or 'loading weights' in lower:
        return '\U0001f504 GPU yükleniyor'
    if 'warming up' in lower or 'cuda graph' in lower:
        return '\U0001f525 CUDA warmup'
    if 'started server' in lower or 'uvicorn running' in lower:
        return '\u2705 Hazır'
    return '\u23f3 Başlatılıyor'


t0 = time.time()
ready = False
while time.time() - t0 < 600:
    elapsed = int(time.time() - t0)
    phase = detect_phase(get_log_tail())
    try:
        if requests.get(f'{VLLM_BASE_URL}/health', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if proc.poll() is not None:
        print(f'\u274c vLLM çöktü! (exit: {proc.returncode})')
        print(get_log_tail(20))
        break
    print(f'[{elapsed:3d}s] {phase}')
    if '\u274c' in phase:
        print(get_log_tail(15))
        break
    time.sleep(5)

if ready:
    print(f'\n\u2705 {ACTIVE_MODEL} hazır! ({int(time.time()-t0)}s)')
    try:
        r = requests.post(
            f'{VLLM_BASE_URL}/v1/chat/completions',
            headers={'Authorization': f'Bearer {API_KEY}'},
            json={'model': MODEL_REPO, 'messages': [{'role': 'user', 'content': 'Say hello'}], 'max_tokens': 32},
            timeout=120,
        )
        if r.status_code == 200:
            msg = r.json()['choices'][0]['message']
            txt = msg.get('content') or msg.get('reasoning') or 'No content'
            print(f'   \U0001f4ac {txt[:150]}')
        else:
            print(f'   \u26a0\ufe0f HTTP {r.status_code}')
    except requests.RequestException as e:
        print(f'   \u26a0\ufe0f {e}')
else:
    print(f'\n\u274c Timeout!\n{get_log_tail(20)}')

---
# C) Cloudflare Tunnel

Bir kere çalıştır. Model değiştiğinde tunnel ayakta kalır.

`api.ersamely.com` \u2192 `localhost:8090`

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata

subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

token = userdata.get('CF_TUNNEL_TOKEN')

CF_LOG = f'{LOG_DIR}/cloudflared.log'
with open(CF_LOG, 'w') as f:
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
        stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
    )

print(f'\U0001f310 Tunnel başlatıldı (PID: {cf_proc.pid})')
time.sleep(5)

try:
    r = requests.get(f'{TUNNEL_URL}/health', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=10)
    if r.status_code == 200:
        print(f'\u2705 Tunnel aktif: {TUNNEL_URL}')
        print(f'\n   Open WebUI:')
        print(f'   URL: {TUNNEL_URL}/v1')
        print(f'   API Key: (Colab Secrets > VLLM_API_KEY)')
    else:
        print(f'\u26a0\ufe0f HTTP {r.status_code}')
except requests.RequestException:
    if cf_proc.poll() is None:
        print(f'\u2705 Tunnel çalışıyor: {TUNNEL_URL}')
        print('   (vLLM hazır olunca erişilebilir)')
    else:
        print('\u274c Tunnel başarısız!')
        try:
            with open(CF_LOG) as f:
                print(f.read()[-500:])
        except OSError:
            pass

In [ ]:
import time
from datetime import datetime, timezone

import requests

print(f'Canlı tutma: {TUNNEL_URL} | Model: {ACTIVE_MODEL}')
print('Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'{VLLM_BASE_URL}/health', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    try:
        tunnel_ok = requests.get(f'{TUNNEL_URL}/health', headers={'Authorization': f'Bearer {API_KEY}'}, timeout=10).status_code == 200
    except requests.RequestException:
        tunnel_ok = False

    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    l = '\u2705' if local_ok else '\u274c'
    t = '\u2705' if tunnel_ok else '\u274c'
    print(f'{now} | {ACTIVE_MODEL} | vLLM: {l} | Tunnel: {t}')
    time.sleep(30)